# Step 1: Import Libraries and Datasets

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn

In [ ]:
from keras.datasets import cifar10
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

In [ ]:
X_train.shape

In [ ]:
X_test.shape

In [ ]:
y_train.shape

In [ ]:
y_test.shape

# Step 2: Visualize Data

In [ ]:
i = 30000
plt.imshow(X_train[i])
print(y_train[i])

In [ ]:
W_grid = 15
L_grid = 15

fig, axes = plt.subplots(L_grid, W_grid, figsize = (25, 25))
axes = axes.ravel()

n_training = len(X_train)

for i in np.arange(0, L_grid * W_grid):
    index = np.random.randint(0, n_training)
    axes[i].imshow(X_train[index])
    axes[i].set_title(y_train[index])
    axes[i].axis('off')

plt.subplots_adjust(hspace=0.4)

In [ ]:
n_training

# Step 3: Data Preparartion

In [ ]:
X_train = X_train.astype('float32')
X_test = X_test.astype('float32')

In [ ]:
number_cat = 10

In [ ]:
y_train

In [ ]:
import keras
y_train = keras.utils.to_categorical(y_train, number_cat)

In [ ]:
y_train

In [ ]:
y_test = keras.utils.to_categorical(y_test, number_cat)

In [ ]:
y_test

In [ ]:
X_train = X_train / 255
X_test = X_test / 255

In [ ]:
X_train

In [ ]:
X_train.shape

In [ ]:
Input_shape = X_train.shape[1:]

In [ ]:
Input_shape

# Step 4: Train The Model

In [ ]:
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, AveragePooling2D, Dense, Flatten, Dropout
from keras.optimizers import Adam
from keras.callbacks import TensorBoard

In [ ]:
cnn_model = Sequential()
cnn_model.add(Conv2D(filters=32, kernel_size=(3, 3), activation='relu', input_shape=Input_shape))
cnn_model.add(Conv2D(filters=32, kernel_size=(3, 3), activation='relu'))
cnn_model.add(MaxPooling2D(2, 2))
cnn_model.add(Dropout(0.3))

cnn_model.add(Conv2D(filters=64, kernel_size=(3, 3), activation='relu'))
cnn_model.add(Conv2D(filters=64, kernel_size=(3, 3), activation='relu'))
cnn_model.add(MaxPooling2D(2, 2))
cnn_model.add(Dropout(0.2))

cnn_model.add(Flatten())

cnn_model.add(Dense(units=512, activation='relu'))

cnn_model.add(Dense(units=512, activation='relu'))

cnn_model.add(Dense(units=10, activation='softmax'))

In [ ]:
from keras.optimizers import RMSprop

cnn_model.compile(
    loss='categorical_crossentropy',
    optimizer=RMSprop(learning_rate=0.001),
    metrics=['accuracy']
)

In [ ]:
history = cnn_model.fit(X_train, y_train, batch_size=32, epochs=2, shuffle=True)

# Step 5: Evaluate The Model

In [ ]:
evaluation = cnn_model.evaluate(X_test, y_test)
print('Test Accuracy: {}'.format(evaluation[1]))

In [ ]:
predictions = cnn_model.predict(X_test)
predicted_class = np.argmax(predictions, axis=1)
predicted_class

In [ ]:
y_test

In [ ]:
y_test = y_test.argmax(1)

In [ ]:
y_test

In [ ]:
L = 7
W = 7
fig, axes = plt.subplots(L, W, figsize=(12, 12))
axes = axes.ravel()

for i in np.arange(0, L * W):
    axes[i].imshow(X_test[i])
    axes[i].set_title('Prediction = {}\nTrue = {}'.format(predicted_class[i], y_test[i]))
    axes[i].axis('off')

plt.subplots_adjust(wspace=1)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(y_test, predicted_class)
cm

plt.figure(figsize = (10, 10))
sns.heatmap(cm, annot=True)

# Step 6: Saving The Model

In [ ]:
import os
directory = os.path.join(os.getcwd(), 'saved_models')

if not os.path.isdir(directory):
    os.makedirs(directory)

model_path = os.path.join(directory, 'keras_cifar10_trained_model.h5')
cnn_model.save(model_path)

# Step 7: Improving The Model With Data Augmentation

## Step 7.1: Minist Dataset Data Augmentation Example

In [ ]:
import keras 
from keras.datasets import cifar10
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

In [ ]:
X_train = X_train.astype('float32')
X_test = X_test.astype('float32')

In [ ]:
X_train.shape

In [ ]:
n = 8
X_train_sample = X_train[:n]

In [ ]:
X_train_sample.shape

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

dataget_train = ImageDataGenerator(rotation_range=90)

# dataget_train = ImageDataGenerator(vertical_flip=True)
# dataget_train = ImageDataGenerator(hright_shift_range=0.5)
# dataget_train = ImageDataGenerator(brightness_range=(1, 5))

dataget_train.fit(X_train_sample)

In [ ]:
from PIL import Image

fig = plt.figure(figsize=(20, 2))  

for x_batch in dataget_train.flow(X_train_sample, batch_size=n):
    for i in range(n):
        ax = fig.add_subplot(1, n, i + 1)
        img = Image.fromarray(x_batch[i].astype('uint8'))  # replaces toimage
        ax.imshow(img)
        ax.axis('off')  
    fig.suptitle('Augmented images (rotated 90 degrees)')
    plt.show()
    break


## Step 7.2: Data Augmentation For The Cifar-10 Dataset

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range=90,
    width_shift_range=0.1,  
    horizontal_flip=True,
    vertical_flip=True
)

In [ ]:
datagen.fit(X_train)

In [ ]:
cnn_model.fit(datagen.flow(X_train, y_train, batch_size = 32), epochs = 2)

In [ ]:
score = cnn_model.evaluate(X_test, y_test)
print('Test Accuracy', score[1])

In [ ]:
# save the model 
directory = os.path.join(os.getcwd(), 'saved_models')

if not os.path.isdir(directory):
    os.makedirs(directory)

model_path = os.path.join(directory, 'keras_cifar10_trained_model_Augmentation.h5')
cnn_model.save(model_path)